### Задание на Четверку (Проведение исследований моделями семантической сегментации)

#### 3. Улучшение бейзлайна

Формирование улучшенного бейзлайна. Скачиваем библиотеки

In [2]:
!pip install segmentation-models-pytorch albumentations opencv-python


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


импорты

In [3]:
import os
import cv2
import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm

c:\Users\2b100\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


пути к данным

In [4]:
IMAGES_DIR = "../train_images"
MASKS_DIR = "../train_labels"

dataset

In [5]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype("float32")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return image, mask.unsqueeze(0)

аугментации

In [6]:
def get_train_transforms():
    return A.Compose([
        A.Resize(384, 384),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(p=0.4),
        A.RandomBrightnessContrast(p=0.4),
        A.GaussNoise(p=0.2),
        A.Normalize(),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(384, 384),
        A.Normalize(),
        ToTensorV2()
    ])

данные

In [7]:
full_dataset = SegmentationDataset(IMAGES_DIR, MASKS_DIR, transform=None)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = get_train_transforms()
val_dataset.dataset.transform = get_val_transforms()

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

c:\Users\2b100\AppData\Local\Programs\Python\Python313\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


CNN (U-Net) модель

In [8]:
modelCNN = smp.Unet(
    encoder_name="resnet50",     
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

Transformer (SegFormer-подобный)

In [9]:
modelTransformer = smp.Unet(
    encoder_name="mit_b2",  
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

Настройка обучения моделей

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelCNN.to(device)
modelTransformer.to(device)

loss_fn = smp.losses.DiceLoss(mode="binary")

optimizerCNN = torch.optim.Adam(modelCNN.parameters(), lr=1e-4)
optimizerTransformer = torch.optim.Adam(modelTransformer.parameters(), lr=1e-4)

Метрики

In [11]:
def iou_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-7) / (union + 1e-7)

def dice_score(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    intersection = (pred * target).sum()
    return (2 * intersection + 1e-7) / (pred.sum() + target.sum() + 1e-7)

def pixel_accuracy(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    correct = (pred == target).float().sum()
    total = target.numel()
    return correct / (total + 1e-7)

Train / Val функции

In [12]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for images, masks in tqdm(loader):
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate(model, loader):
    model.eval()
    iou_total = 0
    dice_total = 0
    acc_total = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            preds = torch.sigmoid(model(images))

            iou_total += iou_score(preds, masks).item()
            dice_total += dice_score(preds, masks).item()
            acc_total += pixel_accuracy(preds, masks).item()

    return iou_total / len(loader), dice_total / len(loader), acc_total / len(loader)

Запуск обучения

In [13]:
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}")

    train_loss_cnn = train_one_epoch(modelCNN, train_loader, optimizerCNN)
    iou_cnn, dice_cnn, acc_cnn = validate(modelCNN, val_loader)

    print(f"CNN -> Loss: {train_loss_cnn:.4f}, IoU: {iou_cnn:.4f}, Dice: {dice_cnn:.4f}, PixelAcc: {acc_cnn:.4f}")

    train_loss_tr = train_one_epoch(modelTransformer, train_loader, optimizerTransformer)
    iou_tr, dice_tr, acc_tr = validate(modelTransformer, val_loader)

    print(f"Transformer -> Loss: {train_loss_tr:.4f}, IoU: {iou_tr:.4f}, Dice: {dice_tr:.4f}, PixelAcc: {acc_tr:.4f}")


Epoch 1


100%|██████████| 15/15 [07:08<00:00, 28.57s/it]


CNN -> Loss: 0.5236, IoU: 0.5425, Dice: 0.7012, PixelAcc: 0.8360


100%|██████████| 15/15 [05:21<00:00, 21.45s/it]


Transformer -> Loss: 0.4841, IoU: 0.5659, Dice: 0.7179, PixelAcc: 0.8461

Epoch 2


...

Transformer -> Loss: 0.1414, IoU: 0.8497, Dice: 0.9179, PixelAcc: 0.9663

Epoch 10


100%|██████████| 15/15 [01:43<00:00,  6.87s/it]


CNN -> Loss: 0.1284, IoU: 0.8312, Dice: 0.9067, PixelAcc: 0.9602


100%|██████████| 15/15 [01:26<00:00,  5.74s/it]


Transformer -> Loss: 0.1391, IoU: 0.8565, Dice: 0.9214, PixelAcc: 0.9677


### Анализ результатов и выводы

Применение улучшенного бейзлайна привело к значительному улучшению качества обеих моделей. Наибольший прирост наблюдается у Transformer-модели, которая достигла лучших значений IoU, Dice и Pixel Accuracy. CNN также показала стабильное улучшение, однако её итоговые показатели остаются ниже Transformer. Это подтверждает более высокую эффективность трансформерных архитектур в задаче семантической сегментации при оптимизированном обучении.